In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from src.set_seed import set_seed

from src.model_factory import build_model

from src.embedding_registry import get_embedding_dir

from src.embeddings import load_grouped_embeddings_from_manifest

from src.grouped_holdout_hpy_tuning import run_grouped_holdout_hyperparameter_tuning

from src.grouped_holdout import (
    train_grouped_holdout_cv,
    print_grouped_holdout_summary,
    save_grouped_holdout_results,
)

from src.lofo import (
    train_lofo_cv,
    print_lofo_fold_summary,
    print_lofo_class_recall_summary,
    summarize_aggregated_oof_metrics,
    aggregated_oof_classification_table,
    plot_oof_confusion_matrix,
    plot_oof_roc_curves,
    plot_lofo_training_curves,
    plot_lofo_primary_metric_curve,
)

from src.training import (
    train_classifier,
    print_final_training_summary,
    summarize_final_in_sample_metrics,
    final_in_sample_classification_table,
    plot_final_training_history,
    plot_final_macro_metrics,
    plot_final_confusion_matrix,
    plot_final_roc_curves,
)

from src.reports import (
    report_final_training_diagnostics,
    report_grouped_holdout_main,
    report_grouped_holdout_compact,
    report_lofo_cv_main,
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND MODEL
# ============================================================

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

model = build_model(cfg)

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# LOAD EMBEDDINGS
# ============================================================

evaluation_mode = cfg["evaluation"]["mode"]

print(f"Evaluation mode: {evaluation_mode}")

embedding_dir = get_embedding_dir(cfg)

embeddings_dict = load_grouped_embeddings_from_manifest(embedding_dir, init_embedder=False)

In [ ]:
# ============================================================
# DEFAULT HYPERPARAMETERS FOR GROUPED HOLDOUT HYPERPARAMETER TUNING
# ============================================================

DEFAULT_HP = {
    "optimizer_lr": 1e-3,
    "optimizer_weight_decay": 1e-2,
    "schedular_patience": 10,
    "warmup_epochs": 5,
    "useWeightedSampler": False,
    "weighted_ce_power": -1.0,
    "num_epochs": 75,
    "patience": 8,
    "batch_size": 64,
}

search_space = {
    "optimizer_lr": [1e-3, 3e-4],
    "optimizer_weight_decay": [1e-2, 1e-5, 1e-4],
    "schedular_patience": [2, 8, 15],
    "warmup_epochs": [0, 5, 10],
    "useWeightedSampler": [True, False],
    "weighted_ce_power": [-1.0, 1.0, 2.0],
    "num_epochs": [75],
    "patience": [10],
    "batch_size": [64],
}

best_hp = run_grouped_holdout_hyperparameter_tuning(search_space, cache_path=RUN_DIR / "grouped_holdout_hyperparameter_tuning/grouped_holdout_hyperparameter_tuning_cache.json", save_path=RUN_DIR / "grouped_holdout_hyperparameter_tuning/best_hyperparameters.json")

In [ ]:
# ============================================================
# GROUPED HOLDOUT WITH BEST HYPERPARAMETERS
# ============================================================

grouped_results, grouped_history = train_grouped_holdout_cv(
    file_meta=embeddings_dict['file_meta'],
    embeddings_by_file=embeddings_dict['embeddings_by_file'],
    model_fn=lambda: model(),
    seq_len=embeddings_dict['seq_len'],
    checkpoint_dir=str(RUN_DIR / "grouped_holdout"),
    **best_hp,
)

In [ ]:
print_grouped_holdout_summary(grouped_results)

save_grouped_holdout_results(
    grouped_results,
    prefix=str(RUN_DIR / "grouped_holdout"),
)

In [ ]:
    
# ============================================================
# LOFO CV
# ============================================================

fold_results, last_history = train_lofo_cv(
    all_embeddings=embeddings_dict['all_embeddings'],
    all_labels=embeddings_dict['all_labels'],
    all_file_idx=embeddings_dict['all_file_idx'],
    all_file_names=embeddings_dict['all_file_names'],
    model_fn=lambda: model(),
    device=device,
    checkpoint_dir=str(RUN_DIR / "lofo_cv"),
    **best_hp,
)

In [ ]:
print_lofo_fold_summary(fold_results, save_path=RUN_DIR / "lofo_cv")
print_lofo_class_recall_summary(fold_results, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "lofo_cv")
summarize_aggregated_oof_metrics(fold_results, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "lofo_cv")
aggregated_oof_classification_table(fold_results, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "lofo_cv")

plot_oof_confusion_matrix(fold_results, class_names=['barrier', 'cation', 'anion'], normalize=True, save_path=RUN_DIR / "lofo_cv")
plot_oof_roc_curves(fold_results, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "lofo_cv")
plot_lofo_training_curves(fold_results, save_path=RUN_DIR / "lofo_cv")
plot_lofo_primary_metric_curve(
    fold_results, metric='val_recall',
    ylabel='Recall (%)',
    title='Mean Held-out-Class Recall Across LOFO Folds',
    save_path=RUN_DIR / "lofo_cv"
)

In [ ]:
# ============================================================
# FINAL MODEL TRAINING
# ============================================================

# Prepare all-data loaders 
all_pid = torch.tensor(embeddings_dict['all_file_idx'], dtype=torch.long)
all_ds  = TensorDataset(embeddings_dict['all_embeddings'], embeddings_dict['all_labels'], all_pid)

class_counts_all = Counter(embeddings_dict['all_labels'].numpy())
counts_list_all  = [class_counts_all.get(i, 1) for i in range(3)]

if best_hp["useWeightedSampler"]:
    sample_weights_all = torch.tensor(
        [1.0 / class_counts_all.get(int(l), 1) for l in embeddings_dict['all_labels'].numpy()],
        dtype=torch.float,
    )
    sampler_all = WeightedRandomSampler(
        sample_weights_all, len(sample_weights_all), replacement=True,
    )
    final_train_loader = DataLoader(
        all_ds,
        batch_size=best_hp["batch_size"],
        sampler=sampler_all,
        drop_last=True,
    )
else:
    final_train_loader = DataLoader(
        all_ds,
        batch_size=best_hp["batch_size"],
        shuffle=True,
        drop_last=True,
    )

# Val loader = train loader (in-sample monitoring only, no early stopping)
final_val_loader = DataLoader(
    all_ds,
    batch_size=best_hp["batch_size"],
    shuffle=False,
    drop_last=False,
    )

criterion_all = (
    make_weighted_ce(counts_list_all, device, power=best_hp.get("weighted_ce_power", 1.0))
    if best_hp["useWeightedCE"]
    else None
)

# Train final model on all data
final_model = model().to(device)

history_final = train_classifier(
    final_model, final_train_loader, final_val_loader,
    num_epochs=50,
    device=device,
    patience=50,
    criterion=criterion_all,
    warmup_epochs=best_hp["warmup_epochs"],
    optimizer_lr=best_hp["optimizer_lr"],
    optimizer_weight_decay=best_hp["optimizer_weight_decay"],
    schedular_patience=best_hp["schedular_patience"],
    checkpoint_path=str(RUN_DIR / "model_best_epoch.pt")
)

In [ ]:
# Disclaimer
print("⚠️ The in-sample macro F1 and AUC are expected to be high (possibly near-perfect) since there's no held-out data — these numbers should not be interpreted as generalisation performance.")

# Display tables
print_final_training_summary(history_final, save_path=RUN_DIR / "final_training")
summarize_final_in_sample_metrics(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")
final_in_sample_classification_table(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")

# Display plots
plot_final_training_history(history_final, save_path=RUN_DIR / "final_training")
plot_final_macro_metrics(history_final, save_path=RUN_DIR / "final_training")
plot_final_confusion_matrix(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")
plot_final_roc_curves(history_final, class_names=['barrier', 'cation', 'anion'], save_path=RUN_DIR / "final_training")

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

torch.save(
    final_model.state_dict(),
    RUN_DIR / "final_model.pt",
)

print("Final model saved.")

In [ ]:
# ============================================================
# SAVE FINAL METADATA
# ============================================================

final_metadata = {
    "experiment_name": cfg["experiment"]["name"],
    "evaluation_mode": evaluation_mode,
    "best_config": BEST_CFG,
}

with open(
    RUN_DIR / "training_metadata.json",
    "w",
) as f:

    json.dump(final_metadata, f, indent=2)

print("Training complete.")

In [ ]:
# ============================================================
# SAVE REPORTS - GROUPED HOLDOUT
# ============================================================

_ = report_grouped_holdout_compact(
    grouped_results,
    save_path=RUN_DIR / "reports",
)

In [ ]:
grouped_report = report_grouped_holdout_main(
    grouped_results,
    class_names=['barrier', 'cation', 'anion'],
    show_training_curves=True,
    show_roc=True,
    show_composition=True,
    save_path=RUN_DIR / "reports",
)

In [ ]:
# ============================================================
# SAVE REPORTS - LOFO CV
# ============================================================

_ = report_lofo_cv_main(
    fold_results,
    class_names=['barrier', 'cation', 'anion'],
    show_training_curves=True,
    show_roc=True,
    save_path=RUN_DIR / "reports",
)

In [ ]:
# ============================================================
# SAVE REPORTS - FINAL TRAINING
# ============================================================

report_final_training_diagnostics(
    history_final,
    class_names=['barrier', 'cation', 'anion'],
    show_confusion=True,
    save_path=RUN_DIR / "reports",
)